In [65]:
import aiohttp
import asyncio
import nest_asyncio
import os
from dotenv import load_dotenv
from typing import List, Dict, Any, Tuple, Union
import json
from github import Github, Auth
from gidgethub.aiohttp import GitHubAPI
from gidgethub import GitHubException
import sys
import base64
from stamina import retry, retry_context
import re
from functools import partial
from textacy import preprocessing
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from qdrant_client.http.models import Distance, VectorParams
import uuid
from searcharray import SearchArray
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langgraph.graph import START, StateGraph
from langchain_core.documents import Document
from typing_extensions import List, TypedDict
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# allow for multiple event loops within Jupyter
nest_asyncio.apply()

# load environment variables
load_dotenv()

import pickle

In [3]:
MAX_CHARACTERS = 30_000
NAMESPACE = uuid.NAMESPACE_URL

def strip_markdown(text: str) -> str:
    """Remove common Markdown syntax; keep inner text and emojis."""
    # Headings: ## Title -> Title
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    # Inline code: `code` -> code
    text = re.sub(r"`([^`]+)`", r"\1", text)
    # Bold/italic: **bold** or *italic* -> bold / italic
    text = re.sub(r"\*{1,2}([^*]+)\*{1,2}", r"\1", text)
    text = re.sub(r"_{1,2}([^_]+)_{1,2}", r"\1", text)
    # Link markup: [text](url) -> text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # Collapse many newlines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def make_normalize_text_pipeline(
    *,
    url_repl: str = "",
    unicode_form: str = "NFC",
):
    """
    Build a textacy pipeline: Markdown/HTML → plain text (emojis kept).

    Args:
        url_repl: Replacement for URLs (default ""). Use "_URL_" to keep a placeholder.
        unicode_form: Unicode normalization form ("NFC", "NFD", "NFKC", "NFKD"). Default "NFC".

    Returns:
        A callable that takes a string and returns preprocessed plain text.
    """
    return preprocessing.make_pipeline(
        strip_markdown,
        preprocessing.remove.html_tags,
        preprocessing.normalize.bullet_points,
        preprocessing.normalize.quotation_marks,
        partial(preprocessing.normalize.unicode, form=unicode_form),
        preprocessing.normalize.whitespace,
    )


# Default pipeline instance
normalize_text_pipeline = make_normalize_text_pipeline()

def repo_to_uuid(repo_name: str) -> str:
    return str(uuid.uuid5(NAMESPACE, repo_name))

def normalize_docs(docs: List[Dict[str, Any]]):
    content_str = ''
    for doc in docs:
        content_str += (doc['content'] + '\n\n')
    truncated = content_str[:MAX_CHARACTERS] if len(content_str) > MAX_CHARACTERS else content_str
    normalized_text = normalize_text_pipeline(truncated)
    return normalized_text


def organize_repo_data(repo_data: List[Dict[Any, Any]]) -> Tuple[List[str], List[Dict[str, Union[str, int]]], List[str]]:
    ids = []
    metadata = []
    docs = []
    for repo in repo_data:
        id = repo_to_uuid(repo['repo'])
        description = repo['description']
        topics = repo['topics']
        doc_source = repo['doc_source']
        stars = repo['stars']
        url = repo['url']
        language = repo['language']
        # append to lists for DB consumption
        ids.append(id)
        metadata.append({
            'id': id,
            'repo': repo['repo'],
            'description': description,
            'topics': topics,
            'language': language,
            'doc_source': doc_source,
            'stars': stars,
            'url': url,
        })
        topics_str = f"Topics: {','.join(topics)}\n"
        docs.append(topics_str + normalize_docs(repo['docs']))
    return ids, metadata, docs

In [2]:
# load cached github data
with open("../data/cached/github_data.pkl", "rb") as f:
    starred_repo_data = pickle.load(f)

In [4]:
ids, metadata, docs = organize_repo_data(starred_repo_data)

In [5]:
metadata[0]

{'id': 'a5ef9890-4c52-5a83-8aa1-c1b266a4f6fb',
 'repo': 'pymc-devs/pytensor',
 'description': 'PyTensor allows you to define, optimize, and efficiently evaluate mathematical expressions involving multi-dimensional arrays.',
 'topics': ['ai',
  'bayesian-inference',
  'computational-science',
  'deep-learning',
  'statistics'],
 'language': 'Python',
 'doc_source': 'readme',
 'stars': 594,
 'url': 'https://github.com/pymc-devs/pytensor'}

In [8]:
# metadata is already in memory after organize_repo_data()
df_repos = pd.DataFrame(metadata)

### Build the BM25 search index over repo metadata

In [10]:
def build_search_field(row: dict) -> str:
    """
    Combine topics (hyphen-split), description, language, and repo name
    into a single BM25-searchable string per repo.
    """
    topics_text = " ".join(t.replace("-", " ") for t in (row["topics"] or []))
    desc_text   = row["description"] or ""
    lang_text   = row["language"] or ""
    # split repo name: "pymc-devs/pytensor" → "pymc devs pytensor"
    repo_text   = row["repo"].replace("/", " ").replace("-", " ").replace("_", " ")
    return f"{topics_text} {desc_text} {lang_text} {repo_text}".lower().strip()

df_repos["searchable"]     = df_repos.apply(build_search_field, axis=1)
df_repos["searchable_idx"] = SearchArray.index(df_repos["searchable"])

print(f"Indexed {len(df_repos)} repos")

2026-03-03 15:11:50,292 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-03 15:11:50,293 - searcharray.indexing - INFO - 0 Batch Start tokenization
2026-03-03 15:11:50,293 - searcharray.indexing - INFO - Tokenizing 2049 documents
2026-03-03 15:11:50,316 - searcharray.indexing - INFO - Tokenization -- vstacking
2026-03-03 15:11:50,317 - searcharray.indexing - INFO - Tokenization -- DONE
2026-03-03 15:11:50,318 - searcharray.indexing - INFO - Inverting docs->terms
2026-03-03 15:11:50,321 - searcharray.indexing - INFO - Encoding positions to bit array
2026-03-03 15:11:50,330 - searcharray.indexing - INFO - Batch tokenization complete
2026-03-03 15:11:50,331 - searcharray.indexing - INFO - (main thread) Processing 1 batch results
2026-03-03 15:11:50,333 - searcharray.indexing - INFO - Indexing from tokenization complete
Indexed 2049 repos


#### Define the relevance scoring function

In [40]:
def get_relevant_repos(df: pd.DataFrame, query: str, min_score: float = 0.1) -> pd.DataFrame:
    """
    BM25-score all repos against a query and return those above min_score,
    sorted by descending relevance. These form the ground truth relevant set
    for Precision@K and RAGAS context recall metrics.

    Tune min_score per query:
      - Niche queries (e.g. "bayesian inference"): 0.1–0.5 (fewer matching repos)
      - Broad queries (e.g. "machine learning"):   1.0–2.0 (many matching repos)
    """
    scores = df["searchable_idx"].array.score(query.split())
    result = df.copy()
    result["bm25_score"] = scores
    relevant = result[result["bm25_score"] > min_score].sort_values(
        "bm25_score", ascending=False
    )
    return relevant[["repo", "url", "topics", "language", "bm25_score"]]

#### Define test queries and generate a ground truth dictionary

In [42]:
# Each query is written in natural language matching how a user would search.
# Queries intentionally span different topic types to stress-test both retrieval
# strategies across varied conceptual searches.
TEST_QUERIES = [
    # probabilistic / statistics
    "bayesian",
    # agent frameworks
    "agents",
    # LLM inference / serving
    "llm",
    # RAG / retrieval
    "rag",
    # evaluation / benchmarks
    "evaluation",
    # full text / lexical search
    "search",
    # data science / ML
    "datascience",
    # async / Python
    "asyncio",
    # recommendation systems
    "recommender",
    # data / datasets
    "dataset",
    # code agents / developer tools
    "developer",
    # graph related repos
    "graph",
    # fuzzy / approximate search
    "approximate",
    # finance
    "finance",
    # biology
    "biology"
]

# Build ground truth: query → list of relevant repo name strings
ground_truth: dict[str, list[str]] = {}

for query in TEST_QUERIES:
    relevant = get_relevant_repos(df_repos, query, min_score=0.5)
    ground_truth[query] = relevant["repo"].tolist()
    print(f"[{len(relevant):3d} relevant]  {query}")

print(f"\nGround truth generated for {len(TEST_QUERIES)} queries.")

[ 33 relevant]  bayesian
[ 48 relevant]  agents
[107 relevant]  llm
[ 36 relevant]  rag
[ 21 relevant]  evaluation
[ 86 relevant]  search
[ 12 relevant]  datascience
[ 41 relevant]  asyncio
[ 54 relevant]  recommender
[ 18 relevant]  dataset
[ 34 relevant]  developer
[ 52 relevant]  graph
[ 11 relevant]  approximate
[ 13 relevant]  finance
[  2 relevant]  biology

Ground truth generated for 15 queries.


In [44]:
print(ground_truth)

{'bayesian': ['avehtari/BDA_course_Aalto', 'cognica-io/bayesian-bm25', 'bambinos/bambi', 'hyosubkim/bayes-toolbox', 'arviz-devs/arviz', 'pymc-devs/pymc-resources', 'dotnet/infer', 'uber/orbit', 'PacktPublishing/Bayesian-Analysis-with-Python-Second-Edition', 'automl/SMAC3', 'bayesian-optimization/BayesianOptimization', 'pymc-devs/pymc', 'HIPS/Spearmint', 'CamDavidsonPilon/Probabilistic-Programming-and-Bayesian-Methods-for-Hackers', 'mckinsey/causalnex', 'AllenDowney/BayesianDecisionAnalysis', 'google/lightweight_mmm', 'kidaufo/StatisticalModeling', 'thomaspinder/GPJax', 'automl/HpBandSter', 'pymc-devs/pytensor', 'automl/auto-sklearn', 'ray-project/tune-sklearn', 'pyro-ppl/numpyro', 'blei-lab/edward', 'modAL-python/modAL', 'scikit-optimize/scikit-optimize', 'hyperactive-project/Hyperactive', 'csinva/imodels', 'SimonBlanke/Gradient-Free-Optimizers', 'py-why/dowhy', 'microsoft/nni', 'BoltzmannEntropy/interviews.ai'], 'agents': ['typedef-ai/fenic', 'google/adk-python', 'strands-agents/sdk-p

In [46]:
# build retrievers for RAG evaluation
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# generate langchain documents from texts, ids and metadata
documents = [
    Document(page_content=text, metadata={**meta, "id": id_})
    for text, id_, meta in zip(docs, ids, metadata)
]

In [66]:
# build BM25 retriever
def preprocess_text(text: str) -> List[str]:
    """borrowed from Ask Brave search for common preprocessing steps before TF-IDF"""
    # Lowercase
    text = text.lower()
    # Remove punctuation and digits
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and short words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

# make sure nltk runs properly
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

bm25_retriever = BM25Retriever.from_documents(
    documents,
    k=10,
    bm25_variant="plus",
    preprocess_func=preprocess_text
)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/abirvabdeb/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/abirvabdeb/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/abirvabdeb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [50]:
# define naive dense vector retriever
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="ask_my_bookmark",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="ask_my_bookmark",
    embedding=embeddings,
)
vector_store.add_documents(documents)
naive_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

In [75]:
# build ensemble retriever which utilizes Reciprocal Rank Fusion under the hood to combine results from both retrievers
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, naive_retriever],
    weights=[0.5,0.5]
)

#### Precision@K

In [54]:
def precision_at_k(retrieved_docs: list, relevant_repos: list, k: int) -> float:
    """
    retrieved_docs : list of LangChain Document objects from invoking a retriever
    relevant_repos : ground_truth[query] — list of relevant repo name strings
    k              : cutoff rank
    """
    top_k       = retrieved_docs[:k]
    top_k_repos = [doc.metadata.get("repo") for doc in top_k]
    hits        = sum(1 for repo in top_k_repos if repo in set(relevant_repos))
    return hits / k if k > 0 else 0.0


def evaluate_retriever_precision(
    retriever,
    ground_truth: dict,
    k_values: list[int] = [5, 10],
    sleep_seconds: float = 0.0,
) -> dict:
    results = {k: {} for k in k_values}

    for query, relevant_repos in ground_truth.items():
        if not relevant_repos:
            continue
        retrieved = retriever.invoke(query)
        for k in k_values:
            results[k][query] = precision_at_k(retrieved, relevant_repos, k)
        if sleep_seconds:
            time.sleep(sleep_seconds)

    summary = {}
    for k in k_values:
        scores = list(results[k].values())
        summary[f"mean_precision_at_{k}"] = sum(scores) / len(scores) if scores else 0.0
        summary[f"per_query_at_{k}"]      = results[k]

    return summary


# Evaluate both retrievers
print("=== Pipeline A: Naive Dense Retriever ===")
naive_eval = evaluate_retriever_precision(naive_retriever, ground_truth)
print(f"  Mean Precision@5  : {naive_eval['mean_precision_at_5']:.3f}")
print(f"  Mean Precision@10 : {naive_eval['mean_precision_at_10']:.3f}")

print("\n=== Pipeline B: Hybrid (BM25 + Dense) ===")
# compression_retriever is built inside the retrieve node — reconstruct it standalone:
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

compressor          = CohereRerank(model="rerank-v3.5")
standalone_hybrid   = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
    search_kwargs={"k": 10}
)

# 7s gap keeps well under the 10 calls/min trial limit
hybrid_eval = evaluate_retriever_precision(standalone_hybrid, ground_truth, sleep_seconds=7.0)
print(f"  Mean Precision@5  : {hybrid_eval['mean_precision_at_5']:.3f}")
print(f"  Mean Precision@10 : {hybrid_eval['mean_precision_at_10']:.3f}")

=== Pipeline A: Naive Dense Retriever ===
  Mean Precision@5  : 0.507
  Mean Precision@10 : 0.420

=== Pipeline B: Hybrid (BM25 + Dense + Cohere Rerank) ===
  Mean Precision@5  : 0.360
  Mean Precision@10 : 0.180


In [57]:
naive_eval

{'mean_precision_at_5': 0.5066666666666667,
 'per_query_at_5': {'bayesian': 1.0,
  'agents': 0.2,
  'llm': 0.6,
  'rag': 0.6,
  'evaluation': 0.2,
  'search': 0.8,
  'datascience': 0.2,
  'asyncio': 1.0,
  'recommender': 1.0,
  'dataset': 0.2,
  'developer': 0.4,
  'graph': 0.2,
  'approximate': 0.6,
  'finance': 0.6,
  'biology': 0.0},
 'mean_precision_at_10': 0.42,
 'per_query_at_10': {'bayesian': 0.9,
  'agents': 0.2,
  'llm': 0.8,
  'rag': 0.3,
  'evaluation': 0.3,
  'search': 0.5,
  'datascience': 0.1,
  'asyncio': 1.0,
  'recommender': 1.0,
  'dataset': 0.1,
  'developer': 0.2,
  'graph': 0.2,
  'approximate': 0.4,
  'finance': 0.3,
  'biology': 0.0}}

In [58]:
hybrid_eval

{'mean_precision_at_5': 0.3600000000000001,
 'per_query_at_5': {'bayesian': 0.4,
  'agents': 0.6,
  'llm': 0.4,
  'rag': 0.6,
  'evaluation': 0.2,
  'search': 0.6,
  'datascience': 0.2,
  'asyncio': 0.6,
  'recommender': 0.6,
  'dataset': 0.2,
  'developer': 0.0,
  'graph': 0.4,
  'approximate': 0.4,
  'finance': 0.2,
  'biology': 0.0},
 'mean_precision_at_10': 0.18000000000000005,
 'per_query_at_10': {'bayesian': 0.2,
  'agents': 0.3,
  'llm': 0.2,
  'rag': 0.3,
  'evaluation': 0.1,
  'search': 0.3,
  'datascience': 0.1,
  'asyncio': 0.3,
  'recommender': 0.3,
  'dataset': 0.1,
  'developer': 0.0,
  'graph': 0.2,
  'approximate': 0.2,
  'finance': 0.1,
  'biology': 0.0}}

In [71]:
# weight BM25 heavier
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, naive_retriever],
    weights=[0.75,0.25]
)
compressor          = CohereRerank(model="rerank-v3.5")
standalone_hybrid   = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
    search_kwargs={"k": 10}
)

# 7s gap keeps well under the 10 calls/min trial limit (updating weights to 0.75 to BM25 and 0.25 to dense vector retrieval)
hybrid_eval = evaluate_retriever_precision(standalone_hybrid, ground_truth, sleep_seconds=7.0)
print(f"  Mean Precision@5  : {hybrid_eval['mean_precision_at_5']:.3f}")
print(f"  Mean Precision@10 : {hybrid_eval['mean_precision_at_10']:.3f}")

  Mean Precision@5  : 0.400
  Mean Precision@10 : 0.200


In [72]:
# weight naive (dense retriever) heavier
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, naive_retriever],
    weights=[0.25,0.75]
)
compressor          = CohereRerank(model="rerank-v3.5")
standalone_hybrid   = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
    search_kwargs={"k": 10}
)

hybrid_eval = evaluate_retriever_precision(standalone_hybrid, ground_truth, sleep_seconds=7.0)
print(f"  Mean Precision@5  : {hybrid_eval['mean_precision_at_5']:.3f}")
print(f"  Mean Precision@10 : {hybrid_eval['mean_precision_at_10']:.3f}")

  Mean Precision@5  : 0.400
  Mean Precision@10 : 0.200


In [73]:
# weight BM25 very heavy
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, naive_retriever],
    weights=[0.9,0.1]
)
compressor          = CohereRerank(model="rerank-v3.5")
standalone_hybrid   = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
    search_kwargs={"k": 10}
)

hybrid_eval = evaluate_retriever_precision(standalone_hybrid, ground_truth, sleep_seconds=7.0)
print(f"  Mean Precision@5  : {hybrid_eval['mean_precision_at_5']:.3f}")
print(f"  Mean Precision@10 : {hybrid_eval['mean_precision_at_10']:.3f}")

  Mean Precision@5  : 0.400
  Mean Precision@10 : 0.200


In [74]:
# weight naive very heavy
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, naive_retriever],
    weights=[0.1,0.9]
)
compressor          = CohereRerank(model="rerank-v3.5")
standalone_hybrid   = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
    search_kwargs={"k": 10}
)

hybrid_eval = evaluate_retriever_precision(standalone_hybrid, ground_truth, sleep_seconds=7.0)
print(f"  Mean Precision@5  : {hybrid_eval['mean_precision_at_5']:.3f}")
print(f"  Mean Precision@10 : {hybrid_eval['mean_precision_at_10']:.3f}")

  Mean Precision@5  : 0.400
  Mean Precision@10 : 0.200


#### Evaluate RAGAS metrics

In [76]:
### build RAG graph
def format_context(docs) -> str:
    chunks = []
    for doc in docs:
        meta = doc.metadata
        topics = meta.get('topics', [])
        if topics:
            topics_str = f"Topics: {','.join(topics)}"
        else:
            topics_str = "N/A"
        chunk = f"""---
Repo: {meta.get('repo', 'N/A')}
URL: {meta.get('url', 'N/A')}
Description: {meta.get('description', 'N/A')}
Topics: {topics_str}
Programming Language: {meta.get('language', 'N/A')}
Stars: {meta.get('stars', 'N/A')}

README excerpt:
{doc.page_content.strip()}
---"""
        chunks.append(chunk)
    return "\n\n".join(chunks)


# define the RAG Pipeline in LangChain
RAG_SYSTEM_PROMPT = """
You are AskMyBookmark, a personal research assistant with access to the user's GitHub starred repositories.

Your job is to help the user discover, recall, and explore repositories they have bookmarked on GitHub. You answer questions by reasoning over the retrieved repository context provided to you — not from your general knowledge of what exists on GitHub.

**Ground rules:**
- Only surface repositories that appear in the retrieved context below. Do not invent or suggest repositories that are not present in the context.
- If no retrieved repositories are relevant to the query, say so honestly and suggest the user try rephrasing or broadening their search.
- You may use your general knowledge to explain a topic or technology, but all repository recommendations must come exclusively from the retrieved context.

**When presenting results:**
- Always include the repository's full name (Repo) as a markdown link to its GitHub URL: [Repo](URL)
- Include a brief description of what the repo does (from the description and topics fields), written in your own words if the original description is terse or absent.
- Explain in 1–2 sentences *why* this repository is relevant to the user's query — this is the most important part.
- Group or rank results by relevance if there are several.
- If useful, note the primary programming language, star count, or topics to help the user evaluate the match.

**Tone:** Conversational, concise, and helpful. Treat the user as a developer who starred these repos intentionally and wants quick, intelligent recall — not a tutorial.

---

Retrieved repository context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(RAG_SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template("{question}"),
])
llm = ChatOpenAI(model="gpt-4.1-nano")

def generate(state):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(messages)
    return {"response" : response.content}

class State(TypedDict):
    question: str
    context: List[Document]
    response: str

def retrieve(state):
    retrieved_docs = naive_retriever.invoke(state["question"])
    return {"context" : retrieved_docs}

# build graph
rag_graph_builder = StateGraph(State).add_sequence([retrieve, generate])
rag_graph_builder.add_edge(START, "retrieve")
naive_rag_graph = rag_graph_builder.compile()

In [77]:
def hybrid_retrieve(state):
    compressor = CohereRerank(model="rerank-v3.5")
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=ensemble_retriever, search_kwargs={"k": 10}
    )
    retrieved_docs = compression_retriever.invoke(state["question"])
    return {"context" : retrieved_docs}

# build hybrid graph
hybrid_rag_graph_builder = StateGraph(State).add_sequence([retrieve, generate])
hybrid_rag_graph_builder.add_edge(START, "hybrid_retrieve")
hybrid_rag_graph = rag_graph_builder.compile()

In [78]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    NoiseSensitivity,
    LLMContextRecall,
    ContextEntityRecall,
)
from ragas import EvaluationDataset, SingleTurnSample
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

ragas_llm        = ChatOpenAI(model="gpt-4.1-mini")
ragas_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def build_ragas_dataset(rag_graph, ground_truth: dict, sleep_seconds: float = 0.0) -> EvaluationDataset:
    """
    Run the full RAG graph for each test query and collect:
      - user_input       : the query
      - retrieved_contexts: list of page_content strings from retrieved docs
      - response         : the generated answer
      - reference        : a simple reference answer built from ground truth repos
    """
    samples = []
    for query, relevant_repos in ground_truth.items():
        if not relevant_repos:
            continue

        result    = rag_graph.invoke({"question": query})
        context   = result.get("context", [])
        response  = result.get("response", "")

        reference = "Relevant repositories: " + ", ".join(relevant_repos[:10])

        samples.append(SingleTurnSample(
            user_input          = query,
            retrieved_contexts  = [doc.page_content for doc in context],
            response            = response,
            reference           = reference,
        ))

        if sleep_seconds:
            time.sleep(sleep_seconds)

    return EvaluationDataset(samples=samples)


# Build datasets for both pipelines
print("Building RAGAS evaluation datasets...")
naive_dataset  = build_ragas_dataset(naive_rag_graph, ground_truth)
hybrid_dataset = build_ragas_dataset(hybrid_rag_graph, ground_truth, sleep_seconds=7.0)

metrics = [
    Faithfulness(),
    ResponseRelevancy(),
    NoiseSensitivity(),
    LLMContextRecall(),
    ContextEntityRecall(),
]

print("\nEvaluating Pipeline A: Naive Dense Retriever...")
naive_ragas_results  = evaluate(
    dataset=naive_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\nEvaluating Pipeline B: Hybrid Retriever...")
hybrid_ragas_results = evaluate(
    dataset=hybrid_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\n=== RAGAS Results: Pipeline A (Naive) ===")
print(naive_ragas_results)

print("\n=== RAGAS Results: Pipeline B (Hybrid) ===")
print(hybrid_ragas_results)

Building RAGAS evaluation datasets...

Evaluating Pipeline A: Naive Dense Retriever...


Evaluating:   0%|          | 0/75 [00:00<?, ?it/s]

Exception raised in Job[2]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[17]: TimeoutError()
Exception raised in Job[22]: TimeoutError()
Exception raised in Job[24]: TimeoutError()
Exception raised in Job[28]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[30]: TimeoutError()
Exception raised in Job[32]: TimeoutError()
Exception raised in Job[34]: TimeoutError()
Exception raised in Job[37]: TimeoutError()
Exception raised in Job[42]: TimeoutError()
Exception raised in Job[47]: TimeoutError()
Exception raised in Job[52]: TimeoutError()
Exception raised in Job[55]: TimeoutError()
Exception raised in Job[57]: TimeoutError()
Exception raised in Job[60]: TimeoutError()
Exception raised in Job[62]: TimeoutError()
Exception raised in Job[63]: TimeoutError()
Exception raised in Job[67]: TimeoutError()



Evaluating Pipeline B: Hybrid Retriever...


Evaluating:   0%|          | 0/75 [00:00<?, ?it/s]

Exception raised in Job[2]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[17]: TimeoutError()
Exception raised in Job[19]: TimeoutError()
Exception raised in Job[22]: TimeoutError()
Exception raised in Job[24]: TimeoutError()
Exception raised in Job[27]: TimeoutError()
Exception raised in Job[28]: TimeoutError()
Exception raised in Job[32]: TimeoutError()
Exception raised in Job[37]: TimeoutError()
Exception raised in Job[42]: TimeoutError()
Exception raised in Job[44]: TimeoutError()
Exception raised in Job[53]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-JGGdFwfdJBvr246yF7XnOsim on tokens per min (TPM): Limit 200000, Used 184133, Requested 18327. Please try again in 738ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
Exception raised in Job[47]: Ti


=== RAGAS Results: Pipeline A (Naive) ===
{'faithfulness': 0.7351, 'answer_relevancy': 0.1661, 'noise_sensitivity_relevant': 0.0000, 'context_recall': 0.3846, 'context_entity_recall': 0.0000}

=== RAGAS Results: Pipeline B (Hybrid) ===
{'faithfulness': 0.7160, 'answer_relevancy': 0.2089, 'noise_sensitivity_relevant': 0.3000, 'context_recall': 0.1667, 'context_entity_recall': 0.0000}
